# Day 4 - Code breaks, block 2: object detection and sanity checks

Short code breaks that go with the slides (3-5 minutes each):
1. **run** the cell and look at the output,
2. **change** the number marked with `# 👈`, run again,
3. open **👀 What you should see** to check.

A **CPU runtime is enough**. You cannot break anything - if something looks strange: *Runtime → Restart session*,
then run the first two cells again.

**Data:** hand-made boxes and random numbers - no download.

## Setup - run these two cells first

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torchvision.ops import box_iou, nms

print("ready - PyTorch", torch.__version__)

The second cell defines small helpers. **Just run it - you do not need to read it.**
- `draw_boxes(boxes, scores, keep)` - draws boxes; the kept ones solid, the removed ones dashed

In [ ]:
def draw_boxes(boxes, scores, keep):
    fig, ax = plt.subplots(figsize=(4, 4))
    for i, (b, s) in enumerate(zip(boxes, scores)):
        kept = i in keep
        ax.add_patch(plt.Rectangle((b[0], b[1]), b[2] - b[0], b[3] - b[1], fill=False, lw=2,
                                   ls="-" if kept else "--", color="C1" if kept else "grey"))
        ax.text(b[0], b[3] + 0.1, f"box {i}: {s:.2f}", color="C1" if kept else "grey")
    ax.set_xlim(0, 10); ax.set_ylim(0, 10); ax.set_aspect("equal")
    plt.show()


print("helpers ready")

## 💻 Code break 1 - Intersection over Union (IoU)
Two boxes of the same size (4 × 4), the second one shifted to the right. Boxes are (x_min, y_min, x_max, y_max).

In [ ]:
SHIFT = 2.0            # 👈 how far box B is shifted to the right (half the width = 2)

box_a = torch.tensor([[0.0, 0.0, 4.0, 4.0]])
box_b = torch.tensor([[SHIFT, 0.0, 4.0 + SHIFT, 4.0]])
iou = box_iou(box_a, box_b)
print("IoU:", round(iou.item(), 3))

🔧 **Change it:**
1. `SHIFT = 0` and `SHIFT = 4` - what are the two extreme cases?
2. Try `SHIFT = 1.0` and `SHIFT = 1.5` - which one gives an IoU close to 0.5?

<details><summary>👀 What you should see</summary>

- shift 2 (half the width): IoU **0.333** - not 0.5! overlap 8, union 16 + 16 - 8 = 24, 8 / 24 = 1/3
- shift 0: **1.0** (identical boxes); shift 4: **0.0** (they only touch)
- shift 1.0: IoU **0.6**; shift 1.5: IoU **0.455** - IoU 0.5 is reached at a shift of about 1.33 (a third of
  the width). A detection must be quite close to count as correct with the usual threshold 0.5
</details>

## 💻 Code break 2 - Non-maximum suppression (NMS)
A detector found the same object several times. NMS keeps the box with the highest score and removes all
boxes that overlap it too much (IoU above the threshold).

In [ ]:
IOU_THRESHOLD = 0.5    # 👈

boxes = torch.tensor([[1.0, 1.0, 5.0, 5.0],          # box 0
                      [1.5, 1.2, 5.3, 5.0],          # box 1: almost the same as box 0
                      [0.8, 0.6, 4.6, 4.4],          # box 2: almost the same as box 0
                      [6.0, 6.0, 9.0, 9.0],          # box 3: a second object
                      [6.5, 6.2, 9.2, 9.1]])         # box 4: almost the same as box 3
scores = torch.tensor([0.90, 0.75, 0.60, 0.80, 0.70])
keep = nms(boxes, scores, IOU_THRESHOLD)
print("kept boxes:", keep.tolist())
draw_boxes(boxes, scores, keep.tolist())

🔧 **Change it:**
1. `IOU_THRESHOLD = 0.9` - how many boxes survive now?
2. `IOU_THRESHOLD = 0.1`

<details><summary>👀 What you should see</summary>

- threshold 0.5: kept **[0, 3]** - one box per object, the one with the highest score
- threshold 0.9: kept **[0, 3, 1, 4, 2]** - all 5 boxes survive (sorted by score), because no pair overlaps more than 90 %
  → the same object is reported several times
- threshold 0.1: kept **[0, 3]** - here the same as 0.5 (the two objects do not overlap at all)
- NMS is the image version of "merge flagged windows in a row into one alarm" (day 3)
</details>

## 💻 Code break 3 - Before you train: can the model learn ONE batch by heart?
Before a long training run: train on a single batch of 16 images for a few seconds. A working model + training
loop must reach a loss close to 0. If not, there is a bug.

In [ ]:
LR = 0.001         # 👈

torch.manual_seed(0)
x_batch = torch.randn(16, 1, 28, 28)             # one batch of 16 "images" (random numbers)
y_batch = torch.randint(0, 10, (16,))            # random labels - nothing to understand, only to memorise
model = nn.Sequential(                           # the small CNN from block 1
    nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
    nn.Flatten(), nn.Linear(32 * 7 * 7, 10))
loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=LR)
for step in range(200):
    loss = loss_fn(model(x_batch), y_batch)
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 50 == 0:                           # print every 50 steps
        print("step", step, "  loss", round(loss.item(), 4))
print("final loss:", round(loss.item(), 4))

🔧 **Change it:**
1. `LR = 0.00001` - does the check still pass in 200 steps?
2. Put a `#` in front of `opt.step()` (and `LR = 0.001` again). What happens?

<details><summary>👀 What you should see</summary>

- LR 0.001: the loss drops from **2.38** to **0.0011** - almost 0, the check passes ✅
- LR 0.00001: only down to **1.83** after 200 steps - a too small learning rate looks like a bug!
- without `opt.step()`: the loss stays at **2.3839** in every step - the weights are never updated. Exactly this
  kind of bug is found by the sanity check in seconds instead of after a long training run.
</details>